In [8]:
import pandas as pd
import os
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from openai import OpenAI
from pydantic import BaseModel
import json

# Load API key
with open("key.txt", "r") as f:
    api_key = f.read().strip()

client = OpenAI(api_key=api_key)


In [9]:
import re

# Arabic tashkeel unicode range
TASHKEEL_REGEX = re.compile(r'[\u0617-\u061A\u064B-\u0652]')

def remove_tashkeel(text):
    if not isinstance(text, str):
        return text
    return re.sub(TASHKEEL_REGEX, '', text)


In [10]:
# Load Datasets
df_pos = pd.read_csv("majaz_dataset.csv")
df_neg = pd.read_csv("non_majaz_random_sample.csv")

TEXT_COL = "البيت_الكامل"

# Ensure we only keep the text column to avoid schema errors
df_pos = df_pos[[TEXT_COL]].copy()
df_neg = df_neg[[TEXT_COL]].copy()

# Assign Labels
df_pos["label"] = 1
df_neg["label"] = 0

# Merge and Clean
df = pd.concat([df_pos, df_neg], ignore_index=True)
df = df.dropna(subset=[TEXT_COL])
df[TEXT_COL] = df[TEXT_COL].apply(remove_tashkeel)

# Split Data (Stratified to maintain class balance)
X_train, X_test, y_train, y_test = train_test_split(
    df[TEXT_COL].values,
    df["label"].values,
    test_size=0.10, 
    stratify=df["label"],
    random_state=42
)

print(f"Training size: {len(X_train)}")
print(f"Testing size: {len(X_test)}")

Training size: 1434
Testing size: 160


In [11]:
# 2. Build the Massive Context String
# ====================================
# We format every single training example to feed into the prompt
train_lines = []
for verse, label in zip(X_train, y_train):
    lbl_str = "مجاز" if label == 1 else "ليس مجازاً"
    # Format: Verse -> Label
    train_lines.append(f"البيت: {verse}\nالحكم: {lbl_str}")

# Join them with separators
TRAIN_CONTEXT = "\n---\n".join(train_lines)

In [12]:
# 3. Define Prompt & Schema
# =========================
class VerseClassification(BaseModel):
    id: int
    reasoning: str 
    is_majaz: int

class BatchResponse(BaseModel):
    results: list[VerseClassification]

SYSTEM_PROMPT = f"""
أنت عالم ضليع في البلاغة العربية والنقد الأدبي. 

لديك أدناه "قاعدة بيانات مرجعية" تحتوي على {len(X_train)} بيت شعري تم تصنيفها بشكل صحيح مسبقاً.
استخدم هذه الأمثلة كمرجع أساسي (Ground Truth) لفهم نمط التصنيف المطلوب بدقة.

### قاعدة البيانات المرجعية (أمثلة تدريبية):
{TRAIN_CONTEXT}
### نهاية قاعدة البيانات المرجعية

المهمة:
صنف الأبيات الجديدة بناءً على فهمك للأنماط في الأمثلة أعلاه.

قواعد التصنيف:
1 = مجاز (استعارة، مجاز مرسل، مجاز عقلي)
0 = ليس مجازاً (حقيقي، وصف مباشر، تشبيه صريح)

"""

def get_predictions_batch(verses, start_index):
    formatted_verses = "\n".join([f"ID {i+start_index}: {v}" for i, v in enumerate(verses)])

    user_content = f"""
    قم بتحليل الأبيات التالية وتصنيفها (1 أو 0) مع ذكر السبب.
    
    الأبيات الجديدة:
    {formatted_verses}
    """

    try:
        completion = client.beta.chat.completions.parse(
            model="gpt-5.2-2025-12-11",
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_content},
            ],
            response_format=BatchResponse,
            temperature=0.0,
        )
        return completion.choices[0].message.parsed.results
    except Exception as e:
        print(f"Error: {e}")
        return []

In [13]:
# 4. Execution Loop
# =================
# NOTE: Since the context is huge (approx 30k tokens), 
# batching is actually more efficient than size=1 to save on network calls.
# I set it to 10 to speed it up, but you can revert to 1 if you prefer.
BATCH_SIZE = 10 
predictions_map = {}

print(f"Starting prediction...")

for i in tqdm(range(0, len(X_test), BATCH_SIZE)):
    batch_verses = X_test[i : i + BATCH_SIZE]
    results = get_predictions_batch(batch_verses, start_index=i)
    
    for res in results:
        predictions_map[res.id] = {"pred": res.is_majaz, "reason": res.reasoning}

Starting prediction...


100%|██████████| 16/16 [04:29<00:00, 16.87s/it]


In [14]:
# 5. Save & Evaluate
# ==================
final_preds = []
final_reasons = []

for i in range(len(X_test)):
    if i in predictions_map:
        final_preds.append(predictions_map[i]["pred"])
        final_reasons.append(predictions_map[i]["reason"])
    else:
        final_preds.append(0)
        final_reasons.append("Error")

print(f"Accuracy: {accuracy_score(y_test, final_preds):.4f}")
print(confusion_matrix(y_test, final_preds))
print(classification_report(y_test, final_preds, target_names=["Non-Majaz", "Majaz"]))

results_df = pd.DataFrame({
    "verse": X_test,
    "y_true": y_test,
    "y_pred": final_preds,
    "reasoning": final_reasons
})

results_df.to_csv("gpt_majaz_full_context_results.csv", index=False, encoding="utf-8-sig")

Accuracy: 0.5813
[[43 33]
 [34 50]]
              precision    recall  f1-score   support

   Non-Majaz       0.56      0.57      0.56        76
       Majaz       0.60      0.60      0.60        84

    accuracy                           0.58       160
   macro avg       0.58      0.58      0.58       160
weighted avg       0.58      0.58      0.58       160

